# 06 — Evaluate All Models on Clear-Day Test Set

Loads the best checkpoint for each model and evaluates on `clear_day/val`.
Also profiles hardware metrics (VRAM, FLOPs, inference speed).
Results are saved to `results/clear_day/`.

**Prerequisites:** notebooks 02–05 must have completed training.

In [ ]:
from pathlib import Path
import json, torch
from dotenv import load_dotenv
from src.eval_utils import compute_map, compute_precision_recall
from src.hardware_utils import measure_vram, count_flops_and_params, measure_inference_speed
from src.train_utils import setup_logging

DRIVE_ROOT  = Path('/content/drive/MyDrive/FON/master_rad/bdd100k_data')
CONFIGS_DIR = Path('/content/data_prepared/configs')
CHECKPOINTS = DRIVE_ROOT / 'checkpoints'
RESULTS_DIR = DRIVE_ROOT / 'results' / 'clear_day'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(DRIVE_ROOT / '.env')
logger = setup_logging()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from ultralytics import YOLO, RTDETR

yolov11 = YOLO(str(CHECKPOINTS / 'yolov11/run/weights/best.pt'))
yolov12 = YOLO(str(CHECKPOINTS / 'yolov12/run/weights/best.pt'))
rtdetr  = RTDETR(str(CHECKPOINTS / 'rtdetr/run/weights/best.pt'))

# TODO: Load RF-DETR checkpoint
# from rfdetr import RFDETRMedium
# rfdetr_model = RFDETRMedium()
# rfdetr_model.load(str(CHECKPOINTS / 'rfdetr/best.pt'))

In [ ]:
scores = {}

for name, model in [('yolov11', yolov11), ('yolov12', yolov12), ('rtdetr', rtdetr)]:
    logger.info(f'Evaluating {name} on clear-day val...')
    results = model.val(
        data=str(CONFIGS_DIR / 'dataset.yaml'),
        split='val',
        device=DEVICE,
    )
    scores[name] = {
        **compute_map(results),
        **compute_precision_recall(results),
    }
    logger.info(f'{name}: {scores[name]}')

# TODO: Add RF-DETR evaluation here once API is confirmed

(RESULTS_DIR / 'scores.json').write_text(json.dumps(scores, indent=2))
print('Saved scores to', RESULTS_DIR / 'scores.json')

In [ ]:
dummy = torch.zeros(1, 3, 640, 640).to(DEVICE)
hw_metrics = {}

for name, model in [('yolov11', yolov11), ('yolov12', yolov12), ('rtdetr', rtdetr)]:
    vram  = measure_vram(model.model, dummy, device=DEVICE)
    speed = measure_inference_speed(model.model, dummy, device=DEVICE)
    hw_metrics[name] = {**vram, **speed}
    logger.info(f'{name} hw: {hw_metrics[name]}')

(RESULTS_DIR / 'hw_metrics.json').write_text(json.dumps(hw_metrics, indent=2))
print('Saved hw_metrics to', RESULTS_DIR / 'hw_metrics.json')